#### Ação: Estratégia de Mesclagem
	1.	Mesclar PIB + Lat/Long + Densidade via cod_mun (mais confiável).
	2.	Mesclar Urbanização via nome_mun, com normalização (caixa alta, remoção de acentos e espaços extras).

In [ ]:
# Dependências
import pandas as pd # Import pandas library for data manipulation and analysis
import unicodedata # Import unicodedata module for working with Unicode character properties (e.g., normalizing text)

In [ ]:
# Instalar pyarrow
# pip install pyarrow

# The selected code pip install pyarrow is a command to install the PyArrow package using pip, 
# Python's package installer. PyArrow is a Python library that provides a way to work with Apache Arrow, 
# which is a cross-language development platform for in-memory data. It's commonly used for efficient data 
# interchange between different systems and languages, and it's particularly useful when working with large 
# datasets in data analysis workflows. When executed in a Jupyter notebook cell with a ! prefix or in a terminal, 
# this command will download and install the PyArrow package from the Python Package Index (PyPI).

In [ ]:
# Função para normalizar nomes de município
def normalizar_nome(nome):
    if pd.isna(nome):
        return ''
    nome = unicodedata.normalize('NFKD', str(nome)).encode('ASCII', 'ignore').decode('ASCII')
    return nome.upper().strip()

In [ ]:
# 1. Carregar arquivos
pib_df = pd.read_excel('PIB dos Municipios-IBGE-2021.xlsx', engine='openpyxl')
latlong_df = pd.read_excel('Municipios_latlong-2022.xlsx', engine='openpyxl')
dens_df = pd.read_excel('IBGE_codmun-nommun-densid-2022.xlsx', engine='openpyxl')
urb_df = pd.read_excel('IBGE-tx_urb-2022.xlsx', engine='openpyxl')
idhm_df = pd.read_excel('IDHM2010-atlasbr.xlsx', engine='openpyxl')

In [ ]:
# 2. Renomear colunas para padronização
pib_df.rename(columns={'Código do Município': 'cod_mun', 'Nome do Município': 'nome_mun'}, inplace=True)
latlong_df.rename(columns={'codigo_ibge': 'cod_mun', 'nome': 'nome_mun'}, inplace=True)
dens_df.rename(columns={'CodMun': 'cod_mun', 'NomeMun': 'nome_mun'}, inplace=True)
urb_df.rename(columns={'CodMun': 'cod_mun', 'NomeMun': 'nome_mun'}, inplace=True)
idhm_df.rename(columns={'CodMun': 'nome_mun', 'UF': 'uf', 'IDHM 2010': 'idhm_2010'}, inplace=True)

In [ ]:
# 3. Padronizar cod_mun e nome_mun nas três primeiras bases
for df in [pib_df, latlong_df, dens_df]:
    df['cod_mun'] = df['cod_mun'].astype(str).str.zfill(7)
    df['nome_mun'] = df['nome_mun'].astype(str).apply(lambda x: unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('ASCII')).str.upper().str.strip()

In [ ]:
# 4. Padronizar nome_mun na base de urbanização
urb_df['nome_mun'] = urb_df['nome_mun'].apply(normalizar_nome)

# Para a base de IDHM (não tem cod_mun)
idhm_df['nome_mun'] = idhm_df['nome_mun'].apply(normalizar_nome)
idhm_df['uf'] = idhm_df['uf'].str.upper().str.strip()

In [ ]:
# 6. Normalizar nome e UF para garantir merges consistentes
merged_df['nome_mun'] = merged_df['nome_mun'].apply(normalizar_nome)
merged_df['Sigla da Unidade da Federação'] = merged_df['Sigla da Unidade da Federação'].str.upper().str.strip()

urb_df['UF'] = urb_df['UF'].str.upper().str.strip()
idhm_df['uf'] = idhm_df['uf'].str.upper().str.strip()

In [ ]:
# 5. Integrar PIB + Latitude/Longitude + Densidade Demográfica via código do município
merged_df = pib_df.merge(latlong_df, on='cod_mun', how='left', suffixes=('', '_latlong'))
merged_df = merged_df.merge(dens_df, on='cod_mun', how='left', suffixes=('', '_dens'))

In [ ]:
# 7. Integrar taxa de urbanização via nome + UF
merged_df = merged_df.merge(
    urb_df[['nome_mun', 'UF', 'TxUrb', 'Urbana', 'Rural']],
    left_on=['nome_mun', 'Sigla da Unidade da Federação'],
    right_on=['nome_mun', 'UF'],
    how='left'
)

In [ ]:
# 8. Integrar IDHM 2010 via nome + UF
base_integrada_ibge = merged_df.merge(
    idhm_df[['nome_mun', 'uf', 'idhm_2010']],
    left_on=['nome_mun', 'Sigla da Unidade da Federação'],
    right_on=['nome_mun', 'uf'],
    how='left'
)

In [ ]:
# 9. Dicionário de renomeação das colunas
renomear_colunas = {
    'Código da Grande Região': 'cod_gd_reg',
    'Nome da Grande Região': 'gde_reg',
    'Código da Unidade da Federação': 'cod_uf',
    'Sigla da Unidade da Federação': 'sigla_uf',
    'Nome da Unidade da Federação': 'nome_uf',
    'nome_mun_x': 'municipio_origem',
    'Região Metropolitana': 'regiao_metrop',
    'Código da Mesorregião': 'cod_meso',
    'Nome da Mesorregião': 'nome_meso',
    'Código da Microrregião': 'cod_micro',
    'Nome da Microrregião': 'nome_micro',
    'Código da Região Geográfica Imediata': 'cod_reg_imed',
    'Nome da Região Geográfica Imediata': 'nome_reg_imed',
    'Município da Região Geográfica Imediata': 'mun_reg_imed',
    'Código da Região Geográfica Intermediária': 'cod_reg_intermed',
    'Nome da Região Geográfica Intermediária': 'nome_reg_intermed',
    'Município da Região Geográfica Intermediária': 'mun_reg_intermed',
    'Código Concentração Urbana': 'cod_conc_urba',
    'Nome Concentração Urbana': 'nome_conc_urb',
    'Tipo Concentração Urbana': 'tipo_conc',
    'Código Arranjo Populacional': 'cod_arr',
    'Nome Arranjo Populacional': 'nome_arr',
    'Hierarquia Urbana': 'hierarquia_urb',
    'Hierarquia Urbana (principais categorias)': 'hier_cat',
    'Código da Região Rural': 'cod_reg_rur',
    'Nome da Região Rural': 'nome_reg_rur',
    'Região rural (segundo classificação do núcleo)': 'tipo_reg_rural',
    'Amazônia Legal': 'amaz_legal',
    'Semiárido': 'semiarido',
    'Cidade-Região de São Paulo': 'cid_reg_sao_paulo',
    'Atividade com maior valor adicionado bruto': 'ativ_mais_valor',
    'Atividade com segundo maior valor adicionado bruto': 'ativ_segundo_valor',
    'Atividade com terceiro maior valor adicionado bruto': 'ativ_terceiro_valor',
    'nome_mun_y': 'mun_completo',
    'Urbana': 'pop_urbana',
    'Rural': 'pop_rural',
    'Total': 'pop_total',
    'Valor adicionado bruto da Agropecuária, \na preços correntes\n(R$ 1.000)': 'vr_agrop',
    'Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)': 'vr_ind',
    'Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)': 'vr_sv',
    'Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)': 'vr_admin_pub',
    'Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)': 'vr_ad_tot',
    'Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)': 'imp_liq',
    'Produto Interno Bruto, \na preços correntes\n(R$ 1.000)': 'pib_total',
    'Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)': 'pib_per_capita'
}

# Aplicar renomeações
base_integrada_ibge .rename(columns=renomear_colunas, inplace=True)

# Listar os nomes das colunas do DataFrame
print(base_integrada_ibge .columns.tolist())
print(base_integrada_ibge .shape)

['cod_gd_reg', 'gde_reg', 'cod_uf', 'sigla_uf', 'nome_uf', 'cod_mun', 'nome_mun', 'regiao_metrop', 'cod_meso', 'nome_meso', 'cod_micro', 'nome_micro', 'cod_reg_imed', 'nome_reg_imed', 'mun_reg_imed', 'cod_reg_intermed', 'nome_reg_intermed', 'mun_reg_intermed', 'cod_conc_urba', 'nome_conc_urb', 'tipo_conc', 'cod_arr', 'nome_arr', 'hierarquia_urb', 'hier_cat', 'cod_reg_rur', 'nome_reg_rur', 'tipo_reg_rural', 'amaz_legal', 'semiarido', 'cid_reg_sao_paulo', 'vr_agrop', 'vr_ind', 'vr_sv', 'vr_admin_pub', 'vr_ad_tot', 'imp_liq', 'pib_total', 'pib_per_capita', 'ativ_mais_valor', 'ativ_segundo_valor', 'ativ_terceiro_valor', 'nome_mun_latlong', 'latitude', 'longitude', 'codigo_uf', 'nome_mun_dens', 'Pop', 'Area', 'DensDem', 'UF', 'TxUrb', 'pop_urbana', 'pop_rural', 'uf', 'idhm_2010']
(5570, 56)


In [ ]:
base_integrada_ibge .head()

,cod_gd_reg,gde_reg,cod_uf,sigla_uf,nome_uf,cod_mun,nome_mun,regiao_metrop,cod_meso,nome_meso,...,nome_mun_dens,Pop,Area,DensDem,UF,TxUrb,pop_urbana,pop_rural,uf,idhm_2010
0,1,Norte,11,RO,Rondônia,1100015,ALTA FLORESTA D'OESTE,NaN,1102,Leste Rondoniense,...,ALTA FLORESTA D'OESTE (RO),21494,7067.127,3.04,RO,0.603471,12971.0,8523,RO,0.641
1,1,Norte,11,RO,Rondônia,1100023,ARIQUEMES,NaN,1102,Leste Rondoniense,...,ARIQUEMES (RO),96833,4426.571,21.88,RO,0.866977,83952.0,12881,RO,0.702
2,1,Norte,11,RO,Rondônia,1100031,CABIXI,NaN,1102,Leste Rondoniense,...,CABIXI (RO),5351,1314.352,4.07,RO,0.531863,2846.0,2505,RO,0.650
3,1,Norte,11,RO,Rondônia,1100049,CACOAL,NaN,1102,Leste Rondoniense,...,CACOAL (RO),86887,3793.000,22.91,RO,0.827592,71907.0,14980,RO,0.718
4,1,Norte,11,RO,Rondônia,1100056,CEREJEIRAS,NaN,1102,Leste Rondoniense,...,CEREJEIRAS (RO),15890,2783.300,5.71,RO,0.888357,14116.0,1774,RO,0.692


In [ ]:
# 10. Salvar o resultado final

# CSV
base_integrada_ibge.to_csv('base_integrada_ibge_municipios.csv', index=False)

# Parquet

# Corrigir problemas de tipo (ex: latitude/longitude como string)
colunas_numericas = [
    'latitude', 'longitude', 'Pop', 'DensDem', 'TxUrb',
    'Urbana', 'Rural', 'pop_urbana', 'pop_rural', 'taxa_urbanizacao',
    'pib_per_capita', 'pib_total', 'valor_agropecuaria', 'valor_industria',
    'valor_servicos', 'valor_admin_publica', 'valor_adicionado_total',
    'idhm_2010'
]

for col in colunas_numericas:
    if col in base_integrada_ibge.columns:
        base_integrada_ibge[col] = pd.to_numeric(base_integrada_ibge[col], errors='coerce')

# Salvar como parquet após a correção
base_integrada_ibge.to_parquet('base_integrada_ibge_municipios.parquet', index=False)

In [1]:
import pandas as pd

# Converter .parquet para .csv
df = pd.read_parquet('base_integrada_ibge_municipios.parquet')
df.to_csv('base_ibge_municipios.csv', index=False)